In [ ]:
!pip install albumentations segmentation-models-pytorch -q

import os
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.patches as mpatches

# Thiết lập thiết bị
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 512
BATCH_SIZE = 8 # Điều chỉnh tùy theo bộ nhớ GPU

In [11]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

print("=== HỆ THỐNG TIỀN XỬ LÝ DỮ LIỆU V4 (HỆ 4 LỚP TỐI ƯU) ===")
TARGET_SIZE = (512, 512)

KAGGLE_INPUT = '/kaggle/input/'
KAGGLE_WORKING = '/kaggle/working/'

# TẠO THƯ MỤC MỚI DÀNH RIÊNG CHO V4 ĐỂ TRÁNH NHẦM LẪN
v4_combined_img_dir = os.path.join(KAGGLE_WORKING, 'dataset_v4/train_combined/images/')
v4_combined_mask_dir = os.path.join(KAGGLE_WORKING, 'dataset_v4/train_combined/masks/')

os.makedirs(v4_combined_img_dir, exist_ok=True)
os.makedirs(v4_combined_mask_dir, exist_ok=True)

print(f"📁 Đã tạo thư mục lưu ảnh: {v4_combined_img_dir}")
print(f"📁 Đã tạo thư mục lưu mask: {v4_combined_mask_dir}")

=== HỆ THỐNG TIỀN XỬ LÝ DỮ LIỆU V4 (HỆ 4 LỚP TỐI ƯU) ===
📁 Đã tạo thư mục lưu ảnh: /kaggle/working/dataset_v4/train_combined/images/
📁 Đã tạo thư mục lưu mask: /kaggle/working/dataset_v4/train_combined/masks/


In [13]:
# ==========================================
# QUY TẮC NHÃN MỚI CHO V4 FINAL MASTER:
# 0: Urban (Đô thị / Đường xá)
# 1: Vegetation (Thực vật: Gộp Nông nghiệp + Rừng)
# 2: Water (Nước)
# 3: Barren (Đất trống: Gộp Đất trống + Đồng cỏ + Nền)
# ==========================================

# 1. Từ điển DeepGlobe (RGB sang 4 Class ID)
# Định dạng màu của OpenCV là (Blue, Green, Red)
dg_color_to_v4_id = {
    (255, 255, 0): 0,   # Cyan -> Urban -> 0
    (0, 255, 255): 1,   # Yellow -> Agriculture -> Gộp thành Vegetation (1)
    (0, 255, 0): 1,     # Green -> Forest -> Gộp thành Vegetation (1)
    (255, 0, 0): 2,     # Blue -> Water -> 2
    (255, 255, 255): 3, # White -> Barren -> Gộp thành Barren (3)
    (255, 0, 255): 3,   # Magenta -> Rangeland -> Gộp thành Barren (3)
    (0, 0, 0): 3        # Black -> Unknown -> Gộp thành Barren (3)
}

# 2. Từ điển LoveDA (1D sang 4 Class ID)
loveda_to_v4_id = {
    2: 0, # Building -> Urban (0)
    3: 0, # Road -> Urban (0)
    6: 1, # Forest -> Vegetation (1)
    7: 1, # Agriculture -> Vegetation (1)
    4: 2, # Water -> Water (2)
    5: 3, # Barren -> Barren (3)
    1: 3  # Background -> Barren (3)
}

print("✅ Đã nạp thành công bộ Quy tắc Hệ 4 Lớp nhãn mới!")

✅ Đã nạp thành công bộ Quy tắc Hệ 4 Lớp nhãn mới!


In [14]:
print("\n[1/2] Đang quét radar tìm DeepGlobe...")
dg_img_paths = []

for root, dirs, files in os.walk(KAGGLE_INPUT):
    if 'deepglobe' in root.lower() and 'train' in root.lower() and 'sample' not in root.lower():
        for file in files:
            if file.endswith('_sat.jpg'):
                dg_img_paths.append(os.path.join(root, file))

if not dg_img_paths:
    print("⚠️ Không tìm thấy ảnh DeepGlobe!")
else:
    print(f"-> Tìm thấy {len(dg_img_paths)} ảnh DeepGlobe. Đang xử lý sang Hệ 4 Lớp...")
    for img_path in tqdm(dg_img_paths):
        img_name = os.path.basename(img_path)
        mask_path = img_path.replace('_sat.jpg', '_mask.png')
        
        img = cv2.imread(img_path)
        mask_bgr = cv2.imread(mask_path)
        if img is None or mask_bgr is None: continue
            
        img_res = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        mask_res = cv2.resize(mask_bgr, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
        
        # Áp dụng từ điển quy đổi 4 lớp của DeepGlobe
        gray_mask = np.full(TARGET_SIZE, 3, dtype=np.uint8) # Mặc định nền là 3 (Barren)
        for color, class_id in dg_color_to_v4_id.items():
            gray_mask[np.all(mask_res == color, axis=-1)] = class_id
            
        cv2.imwrite(os.path.join(v4_combined_img_dir, f"DG_{img_name}"), img_res)
        cv2.imwrite(os.path.join(v4_combined_mask_dir, f"DG_{img_name.replace('_sat.jpg', '_mask.png')}"), gray_mask)


[1/2] Đang quét radar tìm DeepGlobe...
-> Tìm thấy 803 ảnh DeepGlobe. Đang xử lý sang Hệ 4 Lớp...


100%|██████████| 803/803 [02:48<00:00,  4.76it/s]


In [15]:
print("\n[2/2] Đang quét radar tìm LoveDA...")
loveda_paths = []

for root, dirs, files in os.walk(KAGGLE_INPUT):
    if 'loveda' in root.lower() and 'train' in root.lower() and 'images_png' in root.lower():
        for file in files:
            if file.endswith('.png'):
                loveda_paths.append(os.path.join(root, file))

if not loveda_paths:
    print("⚠️ Không tìm thấy ảnh LoveDA!")
else:
    print(f"-> Tìm thấy {len(loveda_paths)} ảnh LoveDA. Đang xử lý sang Hệ 4 Lớp...")
    for img_path in tqdm(loveda_paths):
        img_name = os.path.basename(img_path)
        mask_path = img_path.replace('images_png', 'masks_png')
        
        img = cv2.imread(img_path)
        mask_gray = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask_gray is None: continue
            
        img_res = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        mask_res = cv2.resize(mask_gray, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
        
        # Áp dụng từ điển quy đổi 4 lớp của LoveDA
        new_mask = np.full_like(mask_res, 3) # Mặc định nền là 3 (Barren)
        for l_id, v4_id in loveda_to_v4_id.items():
            new_mask[mask_res == l_id] = v4_id
            
        cv2.imwrite(os.path.join(v4_combined_img_dir, f"LDA_{img_name.replace('.png', '.jpg')}"), img_res)
        cv2.imwrite(os.path.join(v4_combined_mask_dir, f"LDA_{img_name}"), new_mask)

print("\n" + "="*50)
print(f"🎉 TẠO DATASET V4 THÀNH CÔNG!")
print(f"📁 Tổng số cặp Ảnh/Mask: {len(os.listdir(v4_combined_img_dir))}")
print("Sẵn sàng để Huấn luyện mô hình V4 Final Master.")
print("="*50)


[2/2] Đang quét radar tìm LoveDA...
-> Tìm thấy 2522 ảnh LoveDA. Đang xử lý sang Hệ 4 Lớp...


100%|██████████| 2522/2522 [02:57<00:00, 14.22it/s]


🎉 TẠO DATASET V4 THÀNH CÔNG!
📁 Tổng số cặp Ảnh/Mask: 3325
Sẵn sàng để Huấn luyện mô hình V4 Final Master.


In [16]:
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
V4_IMG_DIR = '/kaggle/working/dataset_v4/train_combined/images/'
V4_MASK_DIR = '/kaggle/working/dataset_v4/train_combined/masks/'

class KaggleV4Dataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        # Vì ta đã chuẩn hóa tên file ở bước trước, việc tìm mask giờ rất đơn giản
        if 'DG_' in img_name:
            mask_name = img_name.replace('_sat.jpg', '_mask.png')
        else:
            mask_name = img_name.replace('.jpg', '.png')
            
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long()

# Tăng cường dữ liệu (Giữ nguyên như V3 vì đang hoạt động rất tốt)
v4_train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Transpose(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

v4_dataset = KaggleV4Dataset(V4_IMG_DIR, V4_MASK_DIR, transform=v4_train_transform)
v4_train_loader = DataLoader(v4_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

print(f"✅ Đã nạp Data Loader V4. Tổng số ảnh huấn luyện: {len(v4_dataset)}")

✅ Đã nạp Data Loader V4. Tổng số ảnh huấn luyện: 3325


In [18]:
import segmentation_models_pytorch as smp

print("🧠 ĐANG TIẾN HÀNH CHUYỂN GIAO TRỌNG SỐ TỪ V3 SANG V4...")

# 1. Khởi tạo mô hình V4 với classes = 4
model_v4 = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None, # Không tải ImageNet, vì ta sẽ dùng não của V3
    in_channels=3,
    classes=4  # <--- HỆ NHÃN MỚI
).to(DEVICE)

# 2. CẬP NHẬT ĐƯỜNG DẪN ĐÚNG CỦA V3 TỪ KAGGLE INPUT
V3_WEIGHTS_PATH = "/kaggle/input/datasets/duyhuyynh/models/unet_resnet50_V3_Master.pth"

if os.path.exists(V3_WEIGHTS_PATH):
    # Tải bộ nhớ của V3 vào RAM
    v3_state_dict = torch.load(V3_WEIGHTS_PATH, map_location=DEVICE)
    
    # Kỹ thuật cốt lõi: Xóa bỏ bộ nhớ của lớp xuất ra 6 class cũ (segmentation_head)
    v4_state_dict = {k: v for k, v in v3_state_dict.items() if "segmentation_head" not in k}
    
    # Nạp bộ nhớ đã lọc vào V4 (strict=False cho phép bỏ qua phần bị thiếu)
    model_v4.load_state_dict(v4_state_dict, strict=False)
    print("✅ Đã kế thừa thành công phần trích xuất ranh giới của V3 Master!")
    print("✅ Đã khởi tạo lại Lớp phân loại (Segmentation Head) cho hệ 4 nhãn!")
else:
    print(f"⚠️ Không tìm thấy file V3 Master tại: {V3_WEIGHTS_PATH}")
    print("Hãy kiểm tra lại tên thư mục datasets bên thanh công cụ bên phải Kaggle!")

🧠 ĐANG TIẾN HÀNH CHUYỂN GIAO TRỌNG SỐ TỪ V3 SANG V4...
✅ Đã kế thừa thành công phần trích xuất ranh giới của V3 Master!
✅ Đã khởi tạo lại Lớp phân loại (Segmentation Head) cho hệ 4 nhãn!


In [20]:
import torch.nn as nn

# 1. CHIẾN THUẬT KHÓA (FREEZE) ENCODER
# ResNet50 đã quá giỏi trong việc tìm ranh giới nhà cửa và dòng sông, KHÔNG ĐƯỢC làm hỏng nó!
print("🔒 Đang khóa ResNet50 Encoder...")
for param in model_v4.encoder.parameters():
    param.requires_grad = False

# Tốc độ học nhỏ (1e-4) chỉ dành cho Decoder và Head
optimizer_v4 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_v4.parameters()), lr=1e-4, weight_decay=1e-4)

# 2. HÀM LOSS TỐI ƯU (FOCAL + DICE)
# Dice: Giữ ranh giới sắc nét (Phát huy điểm mạnh của V3)
# Focal: Xử lý các ca khó như bóng râm hoặc lùm cây nhỏ (Chữa điểm yếu của V3)
criterion_dice = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
criterion_focal = smp.losses.FocalLoss(mode='multiclass')

def v4_combo_loss(pred, target):
    return 0.5 * criterion_dice(pred, target) + 0.5 * criterion_focal(pred, target)

print("✅ Đã thiết lập xong Hàm Loss Sát Thủ (Focal + Dice) và Optimizer!")

🔒 Đang khóa ResNet50 Encoder...
✅ Đã thiết lập xong Hàm Loss Sát Thủ (Focal + Dice) và Optimizer!


In [21]:
from tqdm import tqdm

V4_EPOCHS = 10 # Chỉ cần học ngắn hạn vì đã có nền tảng V3

print(f"🚀 BẮT ĐẦU FINE-TUNING V4 FINAL MASTER ({V4_EPOCHS} EPOCHS)...")

for epoch in range(V4_EPOCHS):
    model_v4.train()
    running_loss = 0
    loop = tqdm(v4_train_loader, desc=f"V4 Epoch {epoch+1}/{V4_EPOCHS}", leave=True)
    
    for images, masks in loop:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer_v4.zero_grad()
        outputs = model_v4(images)
        loss = v4_combo_loss(outputs, masks)
        
        loss.backward()
        optimizer_v4.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

# LƯU LẠI SIÊU PHẨM
torch.save(model_v4.state_dict(), "/kaggle/working/unet_resnet50_V4_Final_Master.pth")
print("\n🎉 HOÀN TẤT! ĐÃ LƯU: unet_resnet50_V4_Final_Master.pth")

🚀 BẮT ĐẦU FINE-TUNING V4 FINAL MASTER (10 EPOCHS)...


V4 Epoch 10/10: 100%|██████████| 416/416 [03:02<00:00,  2.28it/s, loss=0.21]  



🎉 HOÀN TẤT! ĐÃ LƯU: unet_resnet50_V4_Final_Master.pth
